# Iceberg Table Setup

Создание базы данных и таблицы Iceberg с партиционированием.

## Режимы запуска:
- **Local** (по умолчанию) — быстро, ~3 сек
- **YARN** — для продакшена: замените `%run` на `%run /home/jovyan/work/init_spark.py yarn`

In [ ]:
%run /home/jovyan/work/init_spark.py

### Шаг 0: Создание базы данных silver

In [ ]:
spark.sql("CREATE DATABASE IF NOT EXISTS silver")
print("Database silver created.")

### Шаг 1: Создание таблицы со скрытым партиционированием

In [ ]:
TABLE = "silver.orders"
LOCATION = "hdfs://namenode:9000/warehouse/silver/orders"

spark.sql("CREATE TABLE IF NOT EXISTS " + TABLE + " (" +
    "order_id STRING, user_id STRING, amount DECIMAL(18,2), created_at TIMESTAMP" +
    ") USING iceberg PARTITIONED BY (days(created_at), bucket(16, user_id))" +
    " LOCATION '" + LOCATION + "'")

spark.sql("ALTER TABLE " + TABLE + " SET TBLPROPERTIES ('format-version' = '2', 'write.format.default' = 'parquet')")

print(f"Table {TABLE} created successfully.")

### Шаг 2: Эволюция партиций — переключение с daily на hourly

In [ ]:
# Эволюция через Iceberg Java API (Spark SQL не поддерживает SET PARTITION SPEC с вложенными скобками)
jvm = spark._jvm
table = spark.catalog.loadTable(TABLE)
java_table = table._jtable

update_spec = java_table.updateSpec()
update_spec.removeField("days(created_at)")
update_spec.addField("hours(created_at)")
update_spec.commit()

print("Partition spec evolved: days(created_at) -> hours(created_at)")

### Верификация

In [ ]:
spark.sql("DESCRIBE TABLE " + TABLE).show(truncate=False)
spark.sql("SHOW CREATE TABLE " + TABLE).show(truncate=False)

In [ ]:
spark.stop()